# Save a European map of BMR on SHERPA resolution

In [1]:
import os
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
EU_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [3]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [4]:
# === Set GBD version ===
GBD_version = "GBD23"

In [5]:
# Load one SHERPA file for lat/lon contraints and resolution
eu_file = "EU_concentration_H_2040.nc"
eu_path = os.path.join(EU_DIR, eu_file)
eu = xr.open_dataarray(eu_path)

# Load country mask
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
mask = xr.open_dataarray(mask_path)

# Crop mask to EU bounding box (with buffer for edge safety)
lat_min, lat_max = float(eu.latitude.min()), float(eu.latitude.max())
lon_min, lon_max = float(eu.longitude.min()), float(eu.longitude.max())
buffer = 0.5

mask_eu = mask.sel(
    lat=slice(lat_min - buffer, lat_max + buffer),
    lon=slice(lon_min - buffer, lon_max + buffer),
)

n_multi_eu = int((mask_eu.sum(dim="country") > 1).sum())
if n_multi_eu > 0:
    print(f"WARNING: {n_multi_eu} cells in the EU bbox claimed by >1 country. "
          "Review before proceeding -- argmax below will silently pick the "
          "first match in the country dimension, which may not be what you want.")

print("REVIEW: only 10 cells affected, keep argmax to pick first match")

REVIEW: only 10 cells affected, keep argmax to pick first match


In [6]:
# Collapse (country, lat, lon) from masks -> single country-index grid
# argmax gives the index of the first country with mask==1 in each cell.
# Cells with no country (all zeros, e.g. ocean) will incorrectly get
# index 0 -- so mask those out explicitly using a coverage count.
country_idx = mask_eu.argmax(dim="country")  # int, shape (lat, lon) at 0.1x0.1
has_country = mask_eu.sum(dim="country") > 0
country_idx = country_idx.where(has_country)  # NaN where no country

# Nearest-neighbor regrid the country-index grid onto EU coords
# This is the only "resampling" step, and it's nearest-neighbor because
# country ID is categorical.
country_idx_eu_res = country_idx.interp(
    lat=eu.latitude, lon=eu.longitude, method="nearest"
)

In [7]:
def build_value_grid(bmr, quantile="mean"):
    bmr_lookup = bmr.sel(quantile=quantile).values  # aligned to bmr.country order
    bmr_country_list = list(bmr.country.values)

    # Build an index-to-value array: for each mask country index, find its
    # BMR value. If a mask country isn't in the BMR list, it gets NaN --
    # check the printed list of missing countries below.
    idx_to_value = np.full(len(country_names), np.nan)
    missing = []
    for i, cname in enumerate(country_names):
        if cname in bmr_country_list:
            idx_to_value[i] = bmr_lookup[bmr_country_list.index(cname)]
        else:
            missing.append(cname)

    if missing:
        print(f"NOTE: {len(missing)} mask countries have no BMR match "
              f"(fine if they're outside Europe): {missing[:10]}{'...' if len(missing) > 10 else ''}")

    flat_idx = country_idx_eu_res.values
    out = np.full(flat_idx.shape, np.nan)
    valid = ~np.isnan(flat_idx)
    out[valid] = idx_to_value[flat_idx[valid].astype(int)]

    return xr.DataArray(
        out, coords={"latitude": eu.latitude, "longitude": eu.longitude},
        dims=["latitude", "longitude"], name=f"BMR_COPD_{quantile}"
    )

In [8]:
for health_VAR in health_vars:
    # Load BMR for health_var
    bmr_file = f"{GBD_version}_BMR_Country_{health_VAR}_newlabels_2015-2019.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    bmr = xr.open_dataarray(bmr_path)

    # Map country index -> country name -> BMR value
    country_names = mask.country.values  # ordered to match argmax indices

    bmr_mean_grid = build_value_grid(bmr, "mean")
    bmr_lower_grid = build_value_grid(bmr, "lower")
    bmr_upper_grid = build_value_grid(bmr, "upper")

    result = xr.Dataset({
        "BMR_mean": bmr_mean_grid,
        "BMR_lower": bmr_lower_grid,
        "BMR_upper": bmr_upper_grid,
    })

    # Confirm shape matches the EU target exactly
    assert result.BMR_mean.shape == eu.shape, "Shape mismatch vs EU target grid!"
    print("Output shape:", result.BMR_mean.shape, "-- matches EU grid:", eu.shape)

    out_file = f"{GBD_version}_BMR_European_Country_Map_{health_VAR}_2015-2019.nc"
    out_path = os.path.join(BMR_DIR, out_file)
    print(f"Saving to {out_path}")
    result.to_netcdf(out_path)

print("All processing complete.")

Output shape: (781, 601) -- matches EU grid: (781, 601)
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_COPD_2015-2019.nc
Output shape: (781, 601) -- matches EU grid: (781, 601)
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_DIABETES_2015-2019.nc
Output shape: (781, 601) -- matches EU grid: (781, 601)
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_ISCHEMIC_HEART_DISEASE_2015-2019.nc
Output shape: (781, 601) -- matches EU grid: (781, 601)
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_LOWER_RESPIRATORY_INFECTIONS_2015-2019.nc
Output shape: (781, 601) -- matches EU grid: (781, 601)
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_LUNG_CANCER_2015-2019.nc
Output shape: (781, 601) -- matches EU grid: (781, 601)
Saving to /glade/work/awells/EU_pm/BMR/GBD23_BMR_European_Country_Map_STROKE_2015-2019.nc
Output shape: (781, 601) -- matches EU grid: (781, 601)
Saving to /glade/work/awe